In [ ]:
import torch
import torch.nn as nn
import numpy as np
import math
from torch.utils.data import DataLoader, Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

import tensorflow as tf
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = \
    tf.keras.datasets.imdb.load_data(num_words=10000)

MAX_LEN = 500

def pad_or_truncate(sequence, max_len):
    if len(sequence) > max_len:
        return sequence[:max_len]
    else:
        return [0] * (max_len - len(sequence)) + sequence

class IMDBDataset(Dataset):
    def __init__(self, X, y, max_len):
        self.X = [pad_or_truncate(x, max_len) for x in X]
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return (torch.tensor(self.X[idx], dtype=torch.long),
                torch.tensor(self.y[idx], dtype=torch.float))

train_dataset = IMDBDataset(X_train_raw, y_train_raw, MAX_LEN)
test_dataset  = IMDBDataset(X_test_raw,  y_test_raw,  MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches : {len(test_loader)}")

Device: cuda


2026-05-27 09:38:46.131299: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779874726.373757      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779874726.448821      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779874727.039772      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779874727.039814      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779874727.039816      57 computation_placer.cc:177] computation placer alr

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train batches: 391
Test batches : 391


In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * 
            (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term) 
        pe[:, 1::2] = torch.cos(position * div_term)  
        
        pe = pe.unsqueeze(0)  
        self.register_buffer('pe', pe)  
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

pe = PositionalEncoding(d_model=128, max_len=500)
test_input = torch.zeros(2, 10, 128) 
output = pe(test_input)
print(f"Input shape : {test_input.shape}")
print(f"Output shape: {output.shape}")  # same shape
print(f"PE adds position info to each word vector")

Input shape : torch.Size([2, 10, 128])
Output shape: torch.Size([2, 10, 128])
PE adds position info to each word vector


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % n_heads == 0  
        
        self.d_model  = d_model
        self.n_heads  = n_heads
        self.d_k      = d_model // n_heads  
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)  
        
        self.dropout = nn.Dropout(0.1)
    
    def split_heads(self, x):
        batch_size, seq_len, _ = x.shape
        x = x.view(batch_size, seq_len, self.n_heads, self.d_k)
        return x.transpose(1, 2)
    
    def attention(self, Q, K, V, mask=None):
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
  
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        output = torch.matmul(attn_weights, V)
        
        return output, attn_weights
    
    def forward(self, x, mask=None):
        batch_size = x.size(0)
        
        Q = self.W_Q(x)  
        K = self.W_K(x)  
        V = self.W_V(x)  
        
        Q = self.split_heads(Q)  
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        attn_output, attn_weights = self.attention(Q, K, V, mask)
        
        attn_output = attn_output.transpose(1, 2)
        attn_output = attn_output.contiguous().view(batch_size, -1, self.d_model)
        
        output = self.W_O(attn_output)
        return output, attn_weights

mha = MultiHeadAttention(d_model=128, n_heads=8)
test_input = torch.zeros(2, 10, 128)  
output, weights = mha(test_input)
print(f"Input shape        : {test_input.shape}")
print(f"Output shape       : {output.shape}")
print(f"Attention weights  : {weights.shape}")
print(f"d_k per head       : {128 // 8} dimensions")

Input shape        : torch.Size([2, 10, 128])
Output shape       : torch.Size([2, 10, 128])
Attention weights  : torch.Size([2, 8, 10, 10])
d_k per head       : 16 dimensions


In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.1):
        super(EncoderBlock, self).__init__()
        
        self.attention  = MultiHeadAttention(d_model, n_heads)
        
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),  
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model)   
        )
        
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        attn_output, _ = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_output))  
        
        ff_output = self.ff(x)
        x = self.norm2(x + self.dropout(ff_output))    
        
        return x

encoder_block = EncoderBlock(d_model=128, n_heads=8, ff_dim=512)
test_input    = torch.zeros(2, 10, 128)
output        = encoder_block(test_input)
print(f"Input shape : {test_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Shape unchanged — can stack multiple blocks")

Input shape : torch.Size([2, 10, 128])
Output shape: torch.Size([2, 10, 128])
Shape unchanged — can stack multiple blocks


In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, 
                 ff_dim, max_len, dropout=0.1):
        super(TransformerClassifier, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(d_model, n_heads, ff_dim, dropout)
            for _ in range(n_layers)
        ])
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        
        mask = (x != 0).unsqueeze(1).unsqueeze(2)
      
        
        x = self.embedding(x)           
        x = self.pos_encoding(x)  
        
        for block in self.encoder_blocks:
            x = block(x, mask)         
        
        x = x.mean(dim=1)           
        
        return self.classifier(x).squeeze(1)


# Initialize model
transformer = TransformerClassifier(
    vocab_size = 10000,
    d_model    = 128,    
    n_heads    = 8,      
    n_layers   = 2,     
    ff_dim     = 512, 
    max_len    = 500,  
    dropout    = 0.1
).to(device)

total_params = sum(p.numel() for p in transformer.parameters())
print(f"Total parameters: {total_params:,}")

test_input = torch.randint(0, 10000, (2, 500)).to(device)
output     = transformer(test_input)
print(f"Input shape : {test_input.shape}")
print(f"Output shape: {output.shape}")  

Total parameters: 1,676,673
Input shape : torch.Size([2, 500])
Output shape: torch.Size([2])


In [14]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct    = 0
    total      = 0
    
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        predictions = model(X_batch).squeeze()
        loss        = criterion(predictions, y_batch)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        predicted   = (torch.sigmoid(predictions) >= 0.5).float()
        correct    += (predicted == y_batch).sum().item()
        total      += y_batch.size(0)
    
    return total_loss / len(loader), 100 * correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct    = 0
    total      = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            predictions = model(X_batch).squeeze()
            loss        = criterion(predictions, y_batch)
            
            total_loss += loss.item()
            predicted   = (torch.sigmoid(predictions) >= 0.5).float()
            correct    += (predicted == y_batch).sum().item()
            total      += y_batch.size(0)
    
    return total_loss / len(loader), 100 * correct / total

In [ ]:
criterion  = nn.BCEWithLogitsLoss()
optimizer  = torch.optim.Adam(transformer.parameters(), lr=0.0001)  
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.5)

n_epochs         = 20
best_val_acc     = 0.0
patience         = 3
patience_counter = 0

print("Training Transformer...")
print("="*50)

for epoch in range(n_epochs):
    train_loss, train_acc = train_epoch(transformer, train_loader,
                                         criterion, optimizer, device)
    val_loss, val_acc     = evaluate(transformer, test_loader,
                                      criterion, device)
    scheduler.step(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        patience_counter = 0
        torch.save(transformer.state_dict(), 'best_transformer.pth')
    else:
        patience_counter += 1
    
    print(f"Epoch {epoch+1:2d}/{n_epochs} | "
          f"Train: {train_acc:.1f}% | "
          f"Val: {val_acc:.1f}% | "
          f"Patience: {patience_counter}/{patience}")
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

transformer.load_state_dict(torch.load('best_transformer.pth'))
print(f"\nTransformer Best Val Accuracy: {best_val_acc:.2f}%")

print(f"\n{'='*45}")
print(f"FINAL COMPARISON")
print(f"{'='*45}")
print(f"Vanilla RNN  : 76.92%")
print(f"LSTM         : 88.13%")
print(f"BiLSTM       : 88.48%")
print(f"Transformer  : {best_val_acc:.2f}%")

Training Transformer...
Epoch  1/20 | Train: 87.9% | Val: 86.0% | Patience: 0/3
Epoch  2/20 | Train: 88.6% | Val: 86.2% | Patience: 0/3
Epoch  3/20 | Train: 88.8% | Val: 86.5% | Patience: 0/3
Epoch  4/20 | Train: 89.2% | Val: 86.6% | Patience: 0/3
Epoch  5/20 | Train: 89.5% | Val: 86.6% | Patience: 0/3
Epoch  6/20 | Train: 89.7% | Val: 86.6% | Patience: 1/3
Epoch  7/20 | Train: 90.3% | Val: 86.0% | Patience: 2/3
Epoch  8/20 | Train: 90.6% | Val: 86.5% | Patience: 3/3
Early stopping at epoch 8

Transformer Best Val Accuracy: 86.59%

FINAL COMPARISON
Vanilla RNN  : 76.92%
LSTM         : 88.13%
BiLSTM       : 88.48%
Transformer  : 86.59%
